# Exploración de conjuntos de datos

In [1]:
from huggingface_hub import login

login()

In [3]:
!hf auth whoami

✓ Logged in
  user: umoqnier
  orgs: ElotlMX,somosnlp-hackathon-2022,somosnlp,UNAMMexico


In [1]:
from datasets import load_dataset, load_dataset_builder

## Nahuatl to spanish

### Information

- Datasource: https://openslr.org/92/
- Glotolog code: `https://glottolog.org/resource/languoid/id/high1278`
- ISO code: `azz` 

Translations and audio paths are embedded into `.eaf` files inside `SpeechTranslations/` we need to first translate those `.eaf` files into `jsonl` manifests

In [12]:
import xml.etree.ElementTree as ET
from pathlib import Path
import os
from tqdm.notebook import tqdm

In [18]:
class EAFParser:
    
    AUDIOS_PATH = "Sound-files-Puebla-Nahuatl"

    def __init__(self, eaf_path):
        self.eaf_path = Path(eaf_path)
        self.tree = ET.parse(eaf_path)
        self.root = self.tree.getroot()
        
        self.time_slots = self._parse_time_slots()
        self.media_file: str | None = self._extract_media_file()

    def _parse_time_slots(self) -> dict:
        """Maps TIME_SLOT_ID (e.g., 'ts1') to TIME_VALUE (ms).
        """
        slots = {}
        time_order = self.root.find("TIME_ORDER")
        if time_order is not None:
            for slot in time_order.findall("TIME_SLOT"):
                slots[slot.get("TIME_SLOT_ID")] = int(slot.get("TIME_VALUE"))
        return slots

    def _extract_media_file(self) -> str | None:
        """Extracts the filename of the associated audio file from the header."""
        header = self.root.find("HEADER")
        if header is not None:
            descriptor = header.find("MEDIA_DESCRIPTOR")
            if descriptor is not None:
                # Extract only the filename from the full URI
                media_url = descriptor.get("MEDIA_URL") or descriptor.get("RELATIVE_MEDIA_URL")
                if media_url:
                    return os.path.basename(media_url)
        return None

    def get_segments(self):
        """
        Parses the tiers and returns a list of segments containing:
        transcription_id, start_time, end_time, transcription_text, and translation_text.
        """
        # 1. Extract all Transcription annotations
        # We look for Tiers where the LINGUISTIC_TYPE_REF is 'Transcripción'
        transcriptions = {}
        for tier in self.root.findall("TIER"):
            ling_type_ref = tier.get("LINGUISTIC_TYPE_REF")
            if ling_type_ref == "Transcripción" or ling_type_ref == "UtteranceType":
                for ann in tier.findall(".//ALIGNABLE_ANNOTATION"):
                    ann_id = ann.get("ANNOTATION_ID")
                    start_ts = ann.get("TIME_SLOT_REF1")
                    end_ts = ann.get("TIME_SLOT_REF2")
                    
                    # Convert time slots to seconds
                    start_time = self.time_slots.get(start_ts, 0) / 1000.0
                    end_time = self.time_slots.get(end_ts, 0) / 1000.0
                    
                    text_val = ann.find("ANNOTATION_VALUE").text if ann.find("ANNOTATION_VALUE") is not None else ""
                    
                    transcriptions[ann_id] = {
                        "start": start_time,
                        "end": end_time,
                        "text": text_val,
                        "translation": None # To be filled by the translation tier
                    }

        # 2. Map Translations to Transcriptions
        # We look for Tiers where the LINGUISTIC_TYPE_REF is 'Traducción'
        for tier in self.root.findall("TIER"):
            if tier.get("LINGUISTIC_TYPE_REF") == "Traducción":
                for ann in tier.findall(".//REF_ANNOTATION"):
                    ref_id = ann.get("ANNOTATION_REF")
                    if ref_id in transcriptions:
                        text_val = ann.find("ANNOTATION_VALUE").text if ann.find("ANNOTATION_VALUE") is not None else ""
                        transcriptions[ref_id]["translation"] = text_val

        # Return only segments that have a translation (since we are building an AST dataset)
        return [
            {
                "audio_file": self.media_file,
                "start": seg["start"],
                "end": seg["end"],
                "duration": seg["end"] - seg["start"],
                "transcription": seg["text"],
                "translation": seg["translation"] 
            }
            for seg in transcriptions.values() if seg["translation"]
        ]

In [19]:
base_dir = Path("azz_data/SpeechTranslation")
all_segments = []

# rglob("*/*.eaf") searches recursively through all subdirectories
for eaf_file in tqdm(base_dir.rglob("*.eaf")):
    try:
        parser = EAFParser(eaf_file)
        segments = parser.get_segments()
        all_segments.extend(segments)
        #print(f"Parsed {eaf_file.name}: found {len(segments)} translated segments")
    except Exception as e:
        print(f"Error parsing {eaf_file}: {e}")

0it [00:00, ?it/s]

In [26]:
all_segments[2]

{'audio_file': 'Tzina_Botan_RMM302_teeyekakiixtia-Campanulaceae_2012-07-23-m.wav',
 'start': 9.509,
 'end': 14.322,
 'duration': 4.812999999999999,
 'transcription': "te:ekaeski:xtia. A:mo n'mati ke:yeh yo:n iwki mono:tsa yo:n xiwit.",
 'translation': 'te:ekaeski:xtia. No sé porque así se llama esa hierba.'}

In [24]:
print(f"Found {len(all_segments)} translated segments in {base_dir}")
print(f"Total audio duration={sum([s["duration"] for s in all_segments]) / 60:.2f}[hrs]")

Found 41047 translated segments in azz_data/SpeechTranslation
Total audio duration=2953.69[hrs]


In [25]:
for i, s in enumerate(all_segments[:10]):
    print(f"Segment {i}: {s['start']}-{s['end']} ({s['duration']:.2f}s) -> {s['translation']}")

Segment 0: 0.0-6.272 (6.27s) -> Bueno Rubén pues ahora platicanos de ese..., esa hierba que florece, se da en los potreros, unos..., 
Segment 1: 6.272-9.509 (3.24s) -> se paran grandes de color rojo, le dicen, yo oigo que le dicen,
Segment 2: 9.509-14.322 (4.81s) -> te:ekaeski:xtia. No sé porque así se llama esa hierba.
Segment 3: 126.835-132.63 (5.80s) -> Y dices, este, la flor a lo mejor también da semillas, yo nunca lo veo.
Segment 4: 196.024-200.145 (4.12s) -> Pues parece que ya es todo, este, nos platicaste de esa flor.
Segment 5: 200.145-203.306 (3.16s) -> ¿Nada más en el potrero, este, se da o se da en otro lugar?
Segment 6: 268.908-273.305 (4.40s) -> Bueno, pues parece que ya es todo, nada más tu nombre y tu pueblo.
Segment 7: 14.322-20.273 (5.95s) -> Pues eso lo llaman esa flor, así como dices
Segment 8: 20.273-26.224 (5.95s) -> m..., pero yo no..., no sé porque lo llaman eskaeski..., te:ekaeski:xtia:ni porque yo nunca lo veo así que.
Segment 9: 26.224-28.336 (2.11s) -> que sa

## Mapuzugun to Spanish (arn-spa)

### Datasource: https://huggingface.co/datasets/mengct00/Mapudungun_iwslt26

**NOTE:** Data download needs manual aggreement to terms of use from HF and been logged in.

In [3]:
MAPUZUGUN_TO_SPANISH = "mengct00/Mapudungun_iwslt26"

### Extractig metadata

In [ ]:
# Load the dataset builder
ds_builder = load_dataset_builder(MAPUZUGUN_TO_SPANISH)

In [ ]:
print(f"Dataset {ds_builder.info.dataset_name}")
print("Features:")
print(", ".join(ds_builder.info.features.keys()))

print("Splits:")
for split, split_info in ds_builder.info.splits.items():
    print(split, split_info.num_examples)

print(f"Dataset size: {ds_builder.info.dataset_size / (1024 ** 2):.2f} MB") 

Dataset mapudungun_iwslt26
Features:
audio, arn, arn_clean, spa
Splits:
train 41092
validation 1201
Dataset size: 24524.27 MB


In [ ]:
map_spa_data = load_dataset(MAPUZUGUN_TO_SPANISH, split="train", streaming=True)

Resolving data files:   0%|          | 0/121 [00:00<?, ?it/s]

In [9]:
data_subset = map_spa_data.take(3)

In [10]:
for i, example in enumerate(data_subset):
    print(f"Example {i}\n")
    audio = example["audio"]
    audio_array = audio["array"]  # shape: (T, )
    print("Audio arr", audio_array)
    sampling_rate = audio["sampling_rate"]
    assert sampling_rate == 44100

    transcript_raw = example["arn"]
    print("Transcript [RAW]", transcript_raw)
    transcript_clean = example["arn_clean"]
    print("Transcript [CLEAN]", transcript_clean)
    translation = example["spa"]
    print("Translation", translation)

Example 0

Audio arr [0.01184082 0.00787354 0.00668335 ... 0.06637573 0.06771851 0.0710144 ]
Transcript [RAW] Marimari lamngen ¿ini pingimi eymi?
Transcript [CLEAN] Marimari lamngen ¿ini pingimi eymi?
Translation Hola hermano, ¿cómo te llamas tú?
Example 1

Audio arr [0.0395813  0.0319519  0.0211792  ... 0.01953125 0.02087402 0.00814819]
Transcript [RAW] Marimari lamngen. Iñche tañi üy Agustin Kurikeo Kintrikeo pingey tañi üy. Iñche ñi mapu Kankura pingi. Feymu ta müli tañi chau, puke nuke, tañi peñi, tañi lamngen. Iñche tañi longko fey Jose <*SPA>sangre Mollfünao pingekefuy, tañi fütache fey kay. Femi tati lamngen.
Transcript [CLEAN] Marimari lamngen. Iñche tañi üy Agustin Kurikeo Kintrikeo pingey tañi üy. Iñche ñi mapu Kankura pingi. Feymu ta müli tañi chau, puke nuke, tañi peñi, tañi lamngen. Iñche tañi longko fey Jose sangre Mollfünao pingekefuy, tañi fütache fey kay. Femi tati lamngen.
Translation Hola hermana. Mi nombre Agustín Curiqueo Quintriqueo se dice mi nombre. Mi tierra se

## Quechua to Spanish (que-spa)

### Datasource: https://github.com/johneortega/IWSLT2026_Quechua_data

**NOTE:** Not sure what is the dataset suitable for *Constrained condition* (systems are trained only on the datasets provided by the organizers )

In [ ]:
import pandas as pd

que_data = pd.read_csv(
    "https://raw.githubusercontent.com/johneortega/IWSLT2026_Quechua_data/refs/heads/main/que_spa_synthetic_translation/train/txt/train.tsv",
    sep="\t",
)

In [10]:
que_data.head()

,path,speaker_id,offset,duration,que,spa
0,quechua_01071.wav,JORGE,0.0,5.51,sulpaykuykikum tayta mario mejía kay maytu ruw...,gracias señor mario mejía por este folleto
1,quechua_02672.wav,MARCOS,0.0,29.05,es nuevo código procesal penal dos bueno seis ...,es nuevo código procesal penal dos bueno seis ...
2,quechua_07287.wav,FELICIA,0.0,30.00,imaymanamanta panelmanta huk trabajo rurarikap...,varios paneles se hará un trabajo sobre esos y...
3,quechua_03506.wav,MARCOS,0.0,28.04,minka minkalla ayni aynilla yanapanakusun kusk...,minka minkalla ayni ayni ayni ayúdennos juntos...
4,quechua_04041.wav,MARCOS,0.0,27.04,hatun wañuyman hatun huchaman dios taytapa rey...,gran muerte gran pecado separación del reino d...


In [11]:
for transcript in que_data.iloc[:10]["que"]:
    print(transcript)

sulpaykuykikum tayta mario mejía kay maytu ruwasqaskirayku
es nuevo código procesal penal dos bueno seis vacantes seis abogados se requiere derecho civil seis liderazgo y transparencia seis ortografía y redacción seis etiqueta social seis abordaje a las victimas de violencia familiar y sexual seis cultura y física seis bueno chayta vacantes nin kaypin arí entonces chaynaqa ahh chaymanta kachkantaq codigo
imaymanamanta panelmanta huk trabajo rurarikapunqa chaykunamantawan carreteramanta chaykunamanta paykuna uyarichkanku napaykurikuni llapan uyarikkunata diez chunka horasta paykuna tomananku caserioman chaytan ñuqa willarikuyta munaskani panay kusami turay chayllachu karichkan napaykuchu pikunapaq napaykullinitaq llapan uyariqkunapaq doctor
minka minkalla ayni aynilla yanapanakusun kuskamanta willarinakusun arí wawqipaniykuna imaynallam imaynallam kaypiña kutirimunchik wawqipaniykuna kaymi kay programa sumaq ayñi radio andahuaylas wayra wasintakama chayayka
hatun wañuyman hatun huchaman

In [12]:
for transcript in que_data.iloc[:10]["spa"]:
    print(transcript)

gracias señor mario mejía por este folleto
es nuevo código procesal penal dos bueno seis vacantes seis abogados requeridos derecho civil seis liderazgo y transparencia seis ortografía y redacción seis etiqueta social seis trato a víctimas de violencia familiar y sexual seis cultura y físico seis bueno esa vacante dice aquí sí entonces ahh entonces también hay código
varios paneles se hará un trabajo sobre esos y el camino de los que están escuchando saludo a todos los oyentes diez diez beben por el pueblo que quiero decirle a mi hermana mi querida hermana eso es todo hay saludos para quien saludos a todos los oyentes doctor
minka minkalla ayni ayni ayni ayúdennos juntos digamos si mis hermanos como están como están de vuelta aquí mis hermanos y hermanas este es el programa hermosa ayni radio andahuaylas ondas que llegan
gran muerte gran pecado separación del reino de dios el padre nos ha llevado al mundo así el pecado de un solo hombre un solo pecado y eso es lo que nos dice romanos ca